# 03 — OCR: TrOCR (Transformer)

Fine-tune `microsoft/trocr-base-printed` (ViT encoder + autoregressive transformer decoder).
Two experiments: wm_polygon, wm_bbox.
Requires: `uv sync --extra transformers --extra cuda` (local GPU) or Colab with transformers installed.

In [ ]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    try:
        from google.colab import drive
        IN_COLAB = True
    except ImportError:
        pass

if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    token = userdata.get("GITHUB_TOKEN", "")
    base = f"https://{token}@github.com" if token else "https://github.com"
    if not Path("/content/WaterMeterCV").exists():
        subprocess.run(["git", "clone", f"{base}/UrranQx/WaterMeterCV.git", "/content/WaterMeterCV"], check=True)
    BRANCH = "feature/ocr-notebooks"
    subprocess.run(["git", "-C", "/content/WaterMeterCV", "checkout", BRANCH], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "rapidfuzz", "shapely", "transformers", "sentencepiece"], check=True)
    ROOT         = Path("/content/WaterMeterCV")
    DATA_ROOT    = Path("/content/drive/MyDrive/WaterMeterCV/WaterMetricsDATA")
    WEIGHTS_BASE = Path("/content/drive/MyDrive/WaterMeterCV/weights")
    RESULTS_DIR  = Path("/content/drive/MyDrive/WaterMeterCV/results")
    WORKERS = 2
else:
    ROOT         = Path("../..")
    DATA_ROOT    = ROOT / "WaterMetricsDATA"
    WEIGHTS_BASE = ROOT / "models/weights"
    RESULTS_DIR  = ROOT / "results"
    WORKERS = 0

sys.path.insert(0, str(ROOT))
WEIGHTS_BASE.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import json, time, csv
import torch, cv2, numpy as np
from datetime import datetime

print(f"ROOT: {ROOT}")
print(f"torch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
MODEL_NAME   = "microsoft/trocr-base-printed"
CROPS_ROOT   = DATA_ROOT / "ocr_crops"
WM_PATH      = DATA_ROOT / "waterMeterDataset/WaterMeters"
WEIGHTS_DIR  = WEIGHTS_BASE / "ocr_trocr"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE    = 384   # TrOCR expected input
EPOCHS      = 10
BATCH_SIZE  = 8
LR          = 5e-5
RUN_NAME    = f"trocr_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
print(f"Device: {DEVICE}")
print(f"Run: {RUN_NAME}")

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm

from models.data.ocr_dataset import prepare_ocr_crops, load_ocr_crops
from models.metrics.evaluation import full_string_accuracy, per_digit_accuracy, character_error_rate

prepare_ocr_crops(WM_PATH, CROPS_ROOT)
processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
print("Ready.")

In [ ]:
class TrOCRDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, label_str = self.samples[idx]
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        pil = Image.fromarray(cv2.resize(img, (IMG_SIZE, IMG_SIZE)))
        pixel_values = processor(images=pil, return_tensors="pt").pixel_values.squeeze(0)
        labels = processor.tokenizer(
            label_str, return_tensors="pt", padding=False
        ).input_ids.squeeze(0)
        return pixel_values, labels


def collate_trocr(batch):
    imgs, labels = zip(*batch)
    pixel_values = torch.stack(imgs)
    max_len = max(l.size(0) for l in labels)
    padded = torch.full((len(labels), max_len), processor.tokenizer.pad_token_id, dtype=torch.long)
    for i, l in enumerate(labels):
        padded[i, :l.size(0)] = l
    padded[padded == processor.tokenizer.pad_token_id] = -100
    return pixel_values, padded


def build_trocr():
    m = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)
    m.config.decoder_start_token_id = processor.tokenizer.cls_token_id
    m.config.pad_token_id           = processor.tokenizer.pad_token_id
    m.config.eos_token_id           = processor.tokenizer.sep_token_id
    m.config.max_length             = 12
    m.config.no_repeat_ngram_size   = 3
    m.config.early_stopping         = True
    m.config.num_beams              = 4
    return m


def train_trocr(model, loader, epochs, device, best_path=None):
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    best_loss = float("inf")
    for epoch in range(epochs):
        model.train(); total = 0.0
        for pixel_values, labels in tqdm(loader, desc=f"epoch {epoch+1}/{epochs}", leave=False):
            pixel_values, labels = pixel_values.to(device), labels.to(device)
            loss = model(pixel_values=pixel_values, labels=labels).loss
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item()
        avg = total / len(loader)
        if best_path and avg < best_loss:
            best_loss = avg; model.save_pretrained(str(best_path))
        if (epoch + 1) % 2 == 0 or epoch == 0:
            print(f"  epoch {epoch+1}/{epochs}  loss={avg:.4f}")
    return model


def orient_fix(pred: str) -> str:
    rev = pred[::-1]
    return rev if rev.isdigit() else pred


@torch.no_grad()
def eval_trocr(model, samples, device):
    model.eval()
    preds, gts, times = [], [], []
    for img_path, label_gt in samples:
        img = cv2.imread(str(img_path))
        pil = Image.fromarray(cv2.cvtColor(cv2.resize(img, (IMG_SIZE, IMG_SIZE)), cv2.COLOR_BGR2RGB))
        pixel_values = processor(images=pil, return_tensors="pt").pixel_values.to(device)
        t0  = time.perf_counter()
        gen = model.generate(pixel_values)
        times.append((time.perf_counter() - t0) * 1000)
        pred_raw = processor.tokenizer.decode(gen[0], skip_special_tokens=True).strip()
        pred = orient_fix("".join(c for c in pred_raw if c.isdigit()))
        preds.append(pred); gts.append(label_gt)
    fsa_raw  = full_string_accuracy(preds, gts)
    fsa_norm = full_string_accuracy([p.lstrip("0") or "0" for p in preds],
                                    [g.lstrip("0") or "0" for g in gts])
    pda = float(np.mean([per_digit_accuracy(p, g) for p, g in zip(preds, gts)]))
    cer = float(np.mean([character_error_rate(p, g) for p, g in zip(preds, gts)]))
    ms  = float(np.mean(times))
    return fsa_raw, fsa_norm, pda, cer, ms, preds, gts


print("TrOCR helpers loaded.")

In [ ]:
wm_poly_train = load_ocr_crops(CROPS_ROOT / "wm_polygon", "train")
wm_poly_test  = load_ocr_crops(CROPS_ROOT / "wm_polygon", "test")
wm_poly_loader = DataLoader(TrOCRDataset(wm_poly_train), batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=WORKERS, collate_fn=collate_trocr)
print(f"WM polygon train={len(wm_poly_train)}, test={len(wm_poly_test)}")
model_wm_poly = build_trocr()
model_wm_poly = train_trocr(model_wm_poly, wm_poly_loader, EPOCHS, DEVICE,
                             WEIGHTS_DIR / f"wm_poly_best_{RUN_NAME}")
model_wm_poly.save_pretrained(str(WEIGHTS_DIR / f"wm_poly_last_{RUN_NAME}"))
wm_poly_fsa_raw, wm_poly_fsa_norm, wm_poly_pda, wm_poly_cer, wm_poly_ms, wm_poly_preds, wm_poly_gts = eval_trocr(model_wm_poly, wm_poly_test, DEVICE)
print(f"WM poly — FSA raw={wm_poly_fsa_raw:.3f} norm={wm_poly_fsa_norm:.3f} PDA={wm_poly_pda:.3f} CER={wm_poly_cer:.3f} {wm_poly_ms:.1f}ms")

In [ ]:
wm_bbox_train = load_ocr_crops(CROPS_ROOT / "wm_bbox", "train")
wm_bbox_test  = load_ocr_crops(CROPS_ROOT / "wm_bbox", "test")
wm_bbox_loader = DataLoader(TrOCRDataset(wm_bbox_train), batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=WORKERS, collate_fn=collate_trocr)
print(f"WM bbox train={len(wm_bbox_train)}, test={len(wm_bbox_test)}")
model_wm_bbox = build_trocr()
model_wm_bbox = train_trocr(model_wm_bbox, wm_bbox_loader, EPOCHS, DEVICE,
                             WEIGHTS_DIR / f"wm_bbox_best_{RUN_NAME}")
model_wm_bbox.save_pretrained(str(WEIGHTS_DIR / f"wm_bbox_last_{RUN_NAME}"))
wm_bbox_fsa_raw, wm_bbox_fsa_norm, wm_bbox_pda, wm_bbox_cer, wm_bbox_ms, wm_bbox_preds, wm_bbox_gts = eval_trocr(model_wm_bbox, wm_bbox_test, DEVICE)
print(f"WM bbox — FSA raw={wm_bbox_fsa_raw:.3f} norm={wm_bbox_fsa_norm:.3f} PDA={wm_bbox_pda:.3f} CER={wm_bbox_cer:.3f} {wm_bbox_ms:.1f}ms")

In [ ]:
print(f"{'='*55}")
print(f"{'Metric':<20} {'WM polygon':>15} {'WM bbox':>15}")
print(f"{'='*55}")
for name, poly_v, bbox_v in [
    ("FSA raw",   wm_poly_fsa_raw,  wm_bbox_fsa_raw),
    ("FSA norm",  wm_poly_fsa_norm, wm_bbox_fsa_norm),
    ("Per-digit", wm_poly_pda,      wm_bbox_pda),
    ("CER",       wm_poly_cer,      wm_bbox_cer),
]:
    print(f"{name:<20} {poly_v:>15.3f} {bbox_v:>15.3f}")
print(f"{'Inference ms':<20} {wm_poly_ms:>15.1f} {wm_bbox_ms:>15.1f}")
print(f"{'='*55}")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
rows = [
    ("WM polygon", CROPS_ROOT / "wm_polygon", wm_poly_preds, wm_poly_gts),
    ("WM bbox",    CROPS_ROOT / "wm_bbox",    wm_bbox_preds, wm_bbox_gts),
]
for row_i, (label, crops_dir, preds, gts) in enumerate(rows):
    samples = load_ocr_crops(crops_dir, "test")
    for col_i, ax in enumerate(axes[row_i]):
        if col_i >= len(samples):
            ax.axis("off"); continue
        img = cv2.imread(str(samples[col_i][0]))
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        gt  = gts[col_i] if col_i < len(gts) else "?"
        pr  = preds[col_i] if col_i < len(preds) else "?"
        ax.set_title(f"GT={gt} Pred={pr}", fontsize=9)
        ax.axis("off")
    axes[row_i][0].set_ylabel(label, fontsize=11)
plt.suptitle("TrOCR Predictions", fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "ocr_trocr_predictions.png", dpi=150)
plt.close()
print("Saved ocr_trocr_predictions.png")

In [ ]:
metrics = {
    "method": "trocr",
    "wm_polygon":     {"fsa_raw": round(wm_poly_fsa_raw,4), "fsa_norm": round(wm_poly_fsa_norm,4),
                       "per_digit_acc": round(wm_poly_pda,4), "cer": round(wm_poly_cer,4),
                       "avg_inference_ms": round(wm_poly_ms,1)},
    "wm_bbox":        {"fsa_raw": round(wm_bbox_fsa_raw,4), "fsa_norm": round(wm_bbox_fsa_norm,4),
                       "per_digit_acc": round(wm_bbox_pda,4), "cer": round(wm_bbox_cer,4),
                       "avg_inference_ms": round(wm_bbox_ms,1)},
    "config": {"epochs": EPOCHS, "batch_size": BATCH_SIZE, "lr": LR,
               "architecture": "TrOCR", "base_model": MODEL_NAME},
    "run_date": datetime.now().isoformat(),
}
with open(RESULTS_DIR / "ocr_trocr_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

csv_path = RESULTS_DIR / "ocr_comparison.csv"
header = ["method",
          "wm_poly_fsa_raw","wm_poly_fsa_norm","wm_poly_pda","wm_poly_cer","wm_poly_ms",
          "wm_bbox_fsa_raw","wm_bbox_fsa_norm","wm_bbox_pda","wm_bbox_cer","wm_bbox_ms","run_date"]
write_hdr = not csv_path.exists()
with open(csv_path, "a", newline="") as f:
    w = csv.writer(f)
    if write_hdr: w.writerow(header)
    w.writerow(["trocr",
                round(wm_poly_fsa_raw,4), round(wm_poly_fsa_norm,4), round(wm_poly_pda,4), round(wm_poly_cer,4), round(wm_poly_ms,1),
                round(wm_bbox_fsa_raw,4), round(wm_bbox_fsa_norm,4), round(wm_bbox_pda,4), round(wm_bbox_cer,4), round(wm_bbox_ms,1),
                datetime.now().strftime("%Y-%m-%d %H:%M")])
print("Results saved.")

## Conclusions

- **WM polygon:** FSA raw=?, FSA norm=?, PDA=?, CER=?, ?ms
- **WM bbox:** FSA raw=?, FSA norm=?, PDA=?, CER=?, ?ms
- **TrOCR vs CNN/CRNN:** fill after running — transformer has stronger prior from pretraining.
- **Inference speed:** TrOCR is significantly slower due to autoregressive decoding.
- **Key finding:** fill after running.